In [29]:
import pandas as pd

In [30]:
movies = pd.read_csv(
    "MovieLens 1M/movies.dat",
    sep="::",
    engine="python",
    encoding="latin-1",
    names= ['MovieID','Title','Genres']
)

ratings = pd.read_csv(
    "MovieLens 1M/ratings.dat",
    sep="::",
    engine="python",
    encoding="latin-1",
    names = ['UserID','MovieID','Rating','Timestamp']
)

users = pd.read_csv(
    "MovieLens 1M/users.dat",
    sep="::",
    engine="python",
    encoding="latin-1",
    names = ['UserID','Gender','Age','Occupation','Zip-code']


)

In [31]:
display(movies.head())
display(ratings.head())
display(users.head())

,MovieID,Title,Genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


,UserID,MovieID,Rating,Timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


,UserID,Gender,Age,Occupation,Zip-code
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455


In [32]:
ratings.drop(columns='Timestamp',inplace=True)

### For the first version of model:

W * X + b1 + b2 + u

* W = user embedding
* X = movie embedding
* b1 = User bias
* b2 = Movie bias
* u = Average movie rating


### I am just going with a user item matrix (row : movies, columns: users, values:rating by user for that movie)

### later we can similarly add features/embeddings like age, occupation to have more patterns.

current plan to incorporate them

(user_emb(already trained) + age_bucket_emb(radnom numbers) + occ_emb(radnom numbers) ) * movie_emb(already trained) + b1 + b2 + u

gradient desecnt/ adam optimzer to find the final embeddings.

**finding movie average rating**

In [33]:
# calculating average ratings for each movie

average_ratings = ratings.groupby('MovieID')[['Rating']].agg('mean')
average_ratings.reset_index(inplace=True)
average_ratings.columns = ['MovieID','avg_rating']


# joining the average ratings to the movies df 

movies = movies.merge(average_ratings, on = ['MovieID'], how='left')

movies.head()

,MovieID,Title,Genres,avg_rating
0,1,Toy Story (1995),Animation|Children's|Comedy,4.146846
1,2,Jumanji (1995),Adventure|Children's|Fantasy,3.201141
2,3,Grumpier Old Men (1995),Comedy|Romance,3.016736
3,4,Waiting to Exhale (1995),Comedy|Drama,2.729412
4,5,Father of the Bride Part II (1995),Comedy,3.006757


In [34]:
len(ratings['MovieID'].unique())

3706

In [35]:
# checking if any missing indexes

movie_ids = ratings['MovieID'].unique()

print("Min:", movie_ids.min())
print("Max:", movie_ids.max())
print("Unique count:", len(movie_ids))
print("Expected count:", movie_ids.max() - movie_ids.min() + 1)



print('-----------------')
# display missing indexes

all_ids = set(range(ratings['MovieID'].min(), ratings['MovieID'].max() + 1))
existing_ids = set(ratings['MovieID'].unique())

missing = sorted(all_ids - existing_ids)

print("Missing IDs:", missing[:20])  # first 20
print("Total missing:", len(missing))

Min: 1
Max: 3952
Unique count: 3706
Expected count: 3952
-----------------
Missing IDs: [51, 91, 109, 115, 143, 221, 284, 285, 323, 395, 399, 400, 403, 604, 620, 622, 625, 629, 636, 646]
Total missing: 246


**We map userId and movieId to continuous 0-based indices so the model can do:**

user_emb[user_idx]
movie_emb[movie_idx]

**to fetch the correct embeddings.**

In [36]:
# first split held-out data BEFORE building maps
all_user_ids  = sorted(ratings['UserID'].unique())
all_movie_ids = sorted(ratings['MovieID'].unique())


# removing some users for implementing retraining for new users 
held_out_users  = all_user_ids[-100:]   # last 100 users
held_out_movies = all_movie_ids[-100:]  # last 100 movies

base_ratings = ratings[
    ~ratings['UserID'].isin(held_out_users) &
    ~ratings['MovieID'].isin(held_out_movies)
].copy()

# NOW build maps from base_ratings only
movie_id_map = {id:i for i,id in enumerate(sorted(base_ratings['MovieID'].unique()))}
user_id_map  = {id:i for i,id in enumerate(sorted(base_ratings['UserID'].unique()))}


# creating this so we can later trace back to original ids
idx2movie = {i: id for id, i in movie_id_map.items()}
idx2user = {i: id for id, i in user_id_map.items()}


base_ratings['user_idx']  = base_ratings['UserID'].map(user_id_map)
base_ratings['movie_idx'] = base_ratings['MovieID'].map(movie_id_map)


In [37]:
print(ratings.shape)
print(base_ratings.shape)

(1000209, 3)
(970277, 5)


**train test split to leave out some random ratings, so we can test the model performance later on unseen ratings**

In [38]:
def per_user_split(ratings_df, test_size=0.2, random_state=42):
    train_list = []
    test_list = []
    
    for user_id, user_ratings in ratings_df.groupby('user_idx'):
        user_ratings = user_ratings.sample(frac=1, random_state=random_state)
        n_test = max(1, int(len(user_ratings) * test_size))
        test_list.append(user_ratings.iloc[:n_test])
        train_list.append(user_ratings.iloc[n_test:])
    
    train = pd.concat(train_list)
    test = pd.concat(test_list)
    
    return train, test

train_df, test_df = per_user_split(base_ratings)


In [39]:
display(train_df.head())
display(test_df.head())

,UserID,MovieID,Rating,user_idx,movie_idx
13,1,2918,4,0,2710
8,1,594,4,0,580
26,1,1097,4,0,1025
6,1,1287,5,0,1195
34,1,1907,4,0,1727


,UserID,MovieID,Rating,user_idx,movie_idx
19,1,2797,4,0,2592
41,1,1961,5,0,1781
47,1,1207,4,0,1117
12,1,2398,4,0,2205
43,1,2692,4,0,2488


In [40]:
user_idx = train_df['user_idx'].to_list()
movie_idx = train_df['movie_idx'].to_list()
rating_vals  = train_df['Rating'].to_list()

In [41]:
print(user_idx[:20])
print(movie_idx[:20])
print(rating_vals[:20])

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[2710, 580, 1025, 1195, 1727, 2162, 957, 2147, 253, 574, 517, 2586, 858, 2483, 2102, 1658, 47, 877, 1104, 964]
[4, 4, 4, 5, 4, 5, 5, 3, 4, 4, 4, 4, 4, 3, 4, 5, 5, 4, 5, 5]


### Now that we have the lists, it is time to work on the training the embeddings 

In [42]:
import tensorflow as tf

from tensorflow import keras

Why regularisation for **user, movie embs** and not **bias embs**??

### My Answer based on my understanding : 

The embedding are supposed to be general and work with all the movie embedggings, so they should not be overfit to certain types of movies. That's why we add l2 penalty so it doesnt get overfit. 

Where as the Bias embs, we actually want to get the pattern that the user may have, like alyways giving more than 2.5 rating, so we dont want to penalise that to make it small.

### Ai answer to refine it:

**On embeddings:**
* A user's embedding needs to work well when dotted with ANY movie's embedding — not just the specific movies they rated in training. L2 prevents the vectors from becoming so large and specific that they only work for training pairs.

**On bias:**
* The bias is supposed to capture exactly that pattern — "this user always rates generously" or "this movie is universally loved." That's real signal, not noise. Penalising it would make the model pretend those patterns don't exist, which makes predictions worse.




**Training**

In [48]:
import numpy as np

# convert to arrays
user_arr = np.array(user_idx)
movie_arr = np.array(movie_idx)
rating_arr = np.array(rating_vals) 

# global mean
global_mean = rating_arr.mean()

user_arr = user_arr.astype('int64')
movie_arr = movie_arr.astype('int64')


rating_arr_normalized = np.array(rating_vals) - global_mean

### Why standardize the ratings?

Without standardization — what the model faces at the start:

Your ratings are 1, 2, 3, 4, 5. Global mean is ~3.58. The model output is:
        
        dot(user_emb, movie_emb) + user_bias + movie_bias

At initialization, embeddings are tiny random numbers near 0 (stddev=0.1). So the dot product starts near 0. Biases also start near 0. So early predictions are all near 0.

But the target is ratings between 1 and 5. The loss on the first batch is huge:
target = 4,  prediction = 0.1  →  loss = (4 - 0.1)² = 15.2

The model panics and makes massive gradient updates trying to get predictions up to the 1-5 range. 

The biases race to catch up first because they're simpler — they just need to become large scalars. 

This is exactly why biases dominated and caused embedding collapse in your case.

With standardization — what changes:
    
    After subtracting mean and dividing by std, targets are centered around 0:
    
    rating 4  →  (4 - 3.58) / 1.116  =  +0.38
    
    rating 2  →  (2 - 3.58) / 1.116  =  -1.41

    rating 5  →  (5 - 3.58) / 1.116  =  +1.27

Now at initialization, predictions are near 0 AND targets are near 0. First batch loss:
target = 0.38,  prediction = 0.1  →  loss = (0.38 - 0.1)² = 0.08

Much smaller loss, much smaller gradient updates. The model makes gentle, stable steps. Biases don't need to race to compensate. Embeddings get a fair chance to learn from the start.

The "balanced starting point" means:

    Model output at init ≈ 0. Target after standardization ≈ 0. They're in the same ballpark from epoch 1. No layer needs to do emergency work to bridge a huge gap — everyone learns at a similar pace.

That's why it helped your model. The embedding collapse was partly caused by the biases racing to bridge the gap between "prediction near 0" and "target near 3.58." Standardization removed that gap entirely.


The gradient math:
For the loss (prediction - target)² the gradients are:
∂loss/∂user_bias  =  2 × (prediction - target)  ← simple, direct
∂loss/∂user_emb   =  2 × (prediction - target) × movie_emb  ← multiplied by another small number
∂loss/∂movie_emb  =  2 × (prediction - target) × user_emb  ← multiplied by another small number
The key difference:
Bias gradient is just the error directly. Embedding gradient is the error multiplied by the other embedding.
At initialization both embeddings are tiny — say 0.05 on average. So:
bias gradient:     2 × (3 - 0.2) = 5.6        ← large update
user_emb gradient: 2 × (3 - 0.2) × 0.05 = 0.28  ← 20x smaller update
movie_emb gradient: 2 × (3 - 0.2) × 0.05 = 0.28  ← 20x smaller update
The bias gets a massive update because nothing is dampening it. The embeddings get a much smaller update because they're scaled by the other tiny embedding.
So early in training:

Bias jumps quickly toward compensating for the error
Embeddings barely move because their gradients are tiny
Bias dominates before embeddings even get a chance to learn



model 1

In [18]:
# # hyperparameters

# num_users  = len(user_id_map)   
# num_movies = len(movie_id_map)  



# num_factors = 15

# # user inputs
# user_input = keras.Input(shape=(1,))
# movie_input = keras.Input(shape=(1,))

# # embeddings
# user_vec  = keras.layers.Embedding(input_dim = num_users, output_dim = num_factors,
#                                   embeddings_regularizer=keras.regularizers.l2(1e-5),
#                                   name= "user_emb")(user_input)

# movie_vec = keras.layers.Embedding(num_movies,num_factors, 
#                                    embeddings_regularizer= keras.regularizers.l2(1e-5),
#                                    name="movie_emb")(movie_input)


# # # adding dropout 
# # user_vec = keras.layers.Dropout(0.2)(user_vec)
# # movie_vec = keras.layers.Dropout(0.2)(movie_vec)



# # biases
# user_bias  = keras.layers.Embedding(num_users,1,name="user_bias")(user_input)
# movie_bias = keras.layers.Embedding(num_movies,1,name = "movie_bias")(movie_input)


# # flatten
# user_vec = keras.layers.Flatten()(user_vec)
# movie_vec = keras.layers.Flatten()(movie_vec)
# user_bias = keras.layers.Flatten()(user_bias)
# movie_bias = keras.layers.Flatten()(movie_bias)

# # dot product + biases
# dot = keras.layers.Dot(axes=1)([user_vec, movie_vec])

# output = keras.layers.Add()([dot, user_bias, movie_bias])


# # build and compile
# model = keras.Model([user_input, movie_input], output)


# model.compile(optimizer=keras.optimizers.Adam(learning_rate = 0.0005),
#               loss = 'mse',
#               metrics=[keras.metrics.RootMeanSquaredError(),'mae'])

# model.summary()

# early_stopping = keras.callbacks.EarlyStopping(
#     monitor='val_loss',
#     patience=5,        # stop if no improvement for 5 epochs
#     restore_best_weights=True  # keep the best model
# )

# # train 
# history = model.fit(
#     [user_arr, movie_arr],
#     rating_arr_normalized,
#     epochs = 100,
#     batch_size = 128,
#     validation_split = 0.2,
#     callbacks=[early_stopping],
#     verbose = 1
# )

model 2 - val loss kept increasing, so quit 

In [19]:
# # hyperparameters
# num_users = max(user_idx) +1
# num_movies = max(movie_idx) +1
# num_factors = 15

# # user inputs
# user_input = keras.Input(shape=(1,))
# movie_input = keras.Input(shape=(1,))

# # embeddings
# user_vec  = keras.layers.Embedding(input_dim = num_users, output_dim = num_factors,
#                                   embeddings_regularizer=keras.regularizers.l2(1e-5), 
#                                   name= "user_emb")(user_input)

# movie_vec = keras.layers.Embedding(num_movies,num_factors, 
#                                    embeddings_regularizer= keras.regularizers.l2(1e-5),
#                                    name="movie_emb")(movie_input)


# # adding dropout 
# user_vec = keras.layers.Dropout(0.2)(user_vec)
# movie_vec = keras.layers.Dropout(0.2)(movie_vec)



# # biases
# user_bias  = keras.layers.Embedding(num_users,1,name="user_bias")(user_input)
# movie_bias = keras.layers.Embedding(num_movies,1,name = "movie_bias")(movie_input)


# # flatten
# user_vec = keras.layers.Flatten()(user_vec)
# movie_vec = keras.layers.Flatten()(movie_vec)
# user_bias = keras.layers.Flatten()(user_bias)
# movie_bias = keras.layers.Flatten()(movie_bias)

# # dot product + biases
# dot = keras.layers.Dot(axes=1)([user_vec, movie_vec])

# output = keras.layers.Add()([dot, user_bias, movie_bias])


# # build and compile
# model_2 = keras.Model([user_input, movie_input], output)



# model_2.compile(optimizer=keras.optimizers.Adam(learning_rate = 0.001),
#               loss = 'mse',
#               metrics=[keras.metrics.RootMeanSquaredError(),'mae'])

# model_2.summary()

# early_stopping = keras.callbacks.EarlyStopping(
#     monitor='val_loss',
#     patience=5,        # stop if no improvement for 5 epochs
#     restore_best_weights=True  # keep the best model
# )

# # train 
# history = model.fit(
#     [user_arr, movie_arr],
#     rating_arr_normalized,
#     epochs = 100,
#     batch_size = 32,
#     validation_split = 0.2,
#     callbacks=[early_stopping],
#     verbose = 1
# )

model 3 with hidden layers

In [20]:
# # hyperparameters
# num_users = max(user_idx) +1
# num_movies = max(movie_idx) +1
# num_factors = 15

# # user inputs
# user_input = keras.Input(shape=(1,))
# movie_input = keras.Input(shape=(1,))

# # embeddings
# user_vec  = keras.layers.Embedding(input_dim = num_users, output_dim = num_factors,
#                                   embeddings_regularizer=keras.regularizers.l2(1e-4), 
#                                   name= "user_emb")(user_input)

# movie_vec = keras.layers.Embedding(num_movies,num_factors, 
#                                    embeddings_regularizer= keras.regularizers.l2(1e-4),
#                                    name="movie_emb")(movie_input)


# # adding dropout 
# user_vec = keras.layers.Dropout(0.3)(user_vec)
# movie_vec = keras.layers.Dropout(0.3)(movie_vec)



# # biases
# user_bias  = keras.layers.Embedding(num_users,1,name="user_bias")(user_input)
# movie_bias = keras.layers.Embedding(num_movies,1,name = "movie_bias")(movie_input)


# # flatten
# user_vec = keras.layers.Flatten()(user_vec)
# movie_vec = keras.layers.Flatten()(movie_vec)
# user_bias = keras.layers.Flatten()(user_bias)
# movie_bias = keras.layers.Flatten()(movie_bias)

# # dot product + biases
# dot = keras.layers.Dot(axes=1)([user_vec, movie_vec])


# x = keras.layers.Add()([dot, user_bias, movie_bias])
# x = keras.layers.Dense(64, activation='elu')(x)
# x = keras.layers.Dense(32, activation='elu')(x)
# x = keras.layers.Dense(16, activation='elu')(x)
# x = keras.layers.Dense(8, activation='elu')(x)
# output = keras.layers.Dense(1)(x) 


# # build and compile
# model_3 = keras.Model([user_input, movie_input], output)


# model_3.compile(optimizer=keras.optimizers.Adam(learning_rate = 0.0005),
#               loss = 'mse',
#               metrics=[keras.metrics.RootMeanSquaredError(),'mae'])

# model_2.summary()

# early_stopping = keras.callbacks.EarlyStopping(
#     monitor='val_loss',
#     patience=5,        # stop if no improvement for 5 epochs
#     restore_best_weights=True  # keep the best model
# )

# # train 
# history_3 = model.fit(
#     [user_arr, movie_arr],
#     rating_arr_normalized,
#     epochs = 100,
#     batch_size = 32,
#     validation_split = 0.2,
#     callbacks=[early_stopping],
#     verbose = 1
# )

Model 5 trying to fix the model as all predictions around mean

**Instead of just subtracting mean, also divide by std**

**Because subtracting global mean didnt do much**

In [49]:
import numpy as np

# convert to arrays
user_arr = np.array(user_idx)
movie_arr = np.array(movie_idx)
rating_arr = np.array(rating_vals) 

# global mean
global_mean = rating_arr.mean()

user_arr = user_arr.astype('int32')
movie_arr = movie_arr.astype('int32')


global_mean = rating_arr.mean()
global_std  = rating_arr.std()
rating_arr_normalized = (rating_arr - global_mean) / global_std # standardized

In [22]:
# # SHUFFLE 
# indices = np.random.permutation(len(user_arr))
# user_arr_s   = user_arr[indices]
# movie_arr_s  = movie_arr[indices]
# rating_arr_s = rating_arr_normalized[indices]


# # hyperparameters

# num_users  = len(user_id_map)   
# num_movies = len(movie_id_map)  



# num_factors = 50

# # user inputs
# user_input = keras.Input(shape=(1,))
# movie_input = keras.Input(shape=(1,))

# # embeddings
# user_vec  = keras.layers.Embedding(input_dim = num_users, output_dim = num_factors,
#                                   embeddings_regularizer=keras.regularizers.l2(1e-6),
#                                   embeddings_initializer=keras.initializers.RandomNormal(mean=0, stddev=0.1),
#                                   name= "user_emb")(user_input)

# movie_vec = keras.layers.Embedding(num_movies,num_factors, 
#                                    embeddings_regularizer= keras.regularizers.l2(1e-6),
#                                    embeddings_initializer=keras.initializers.RandomNormal(mean=0, stddev=0.1),
#                                    name="movie_emb")(movie_input)




# # # adding dropout 
# # user_vec = keras.layers.Dropout(0.2)(user_vec)
# # movie_vec = keras.layers.Dropout(0.2)(movie_vec)



# # biases
# user_bias  = keras.layers.Embedding(num_users,1,
#                                     # embeddings_regularizer= keras.regularizers.l2(1e-5),
#                                     name="user_bias")(user_input)

# movie_bias = keras.layers.Embedding(num_movies,1,
#                                     # embeddings_regularizer= keras.regularizers.l2(1e-5),
#                                     name = "movie_bias")(movie_input)


# # flatten
# user_vec = keras.layers.Flatten()(user_vec)
# movie_vec = keras.layers.Flatten()(movie_vec)
# user_bias = keras.layers.Flatten()(user_bias)
# movie_bias = keras.layers.Flatten()(movie_bias)

# # dot product + biases
# dot = keras.layers.Dot(axes=1)([user_vec, movie_vec])
# # output = keras.layers.Dot(axes=1)([user_vec, movie_vec])


# # output = keras.layers.Add()([dot, user_bias])
# output = keras.layers.Add()([dot, user_bias, movie_bias])


# # build and compile
# model = keras.Model([user_input, movie_input], output)


# model.compile(optimizer=keras.optimizers.Adam(learning_rate = 0.0001),
#               loss = 'mse',
#               metrics=[keras.metrics.RootMeanSquaredError(),'mae'])

# model.summary()

# early_stopping = keras.callbacks.EarlyStopping(
#     monitor='val_loss',
#     patience=15,        # stop if no improvement for 5 epochs
#     restore_best_weights=True  # keep the best model
# )

# # train 
# history = model.fit(
#      [user_arr_s, movie_arr_s],    
#     rating_arr_s,
#     epochs = 150,
#     batch_size = 512,
#     validation_split = 0.3,
#     callbacks=[early_stopping],
#     verbose = 1
# )




In [18]:
# # SHUFFLE - so, during training validation split doesnt get the users or movies from the bottom that are not seen during training
# indices = np.random.permutation(len(user_arr))
# user_arr_s   = user_arr[indices]
# movie_arr_s  = movie_arr[indices]
# rating_arr_s = rating_arr_normalized[indices]

# # hyperparameters

# num_users  = len(user_id_map)   
# num_movies = len(movie_id_map)  

# num_factors = 25

# # user inputs
# user_input = keras.Input(shape=(1,))
# movie_input = keras.Input(shape=(1,))

# # embeddings
# user_vec  = keras.layers.Embedding(input_dim = num_users, output_dim = num_factors,
#                                   embeddings_regularizer=keras.regularizers.l2(1e-6),
#                                   embeddings_initializer=keras.initializers.RandomNormal(mean=0, stddev=0.1),
#                                   name= "user_emb")(user_input)

# movie_vec = keras.layers.Embedding(num_movies,num_factors, 
#                                    embeddings_regularizer= keras.regularizers.l2(1e-6),
#                                    embeddings_initializer=keras.initializers.RandomNormal(mean=0, stddev=0.1),
#                                    name="movie_emb")(movie_input)



# # # model was getting too generalized so removed drop out
# # # adding dropout 
# # user_vec = keras.layers.Dropout(0.2)(user_vec)
# # movie_vec = keras.layers.Dropout(0.2)(movie_vec)



# # biases
# user_bias  = keras.layers.Embedding(num_users,1,
#                                     # embeddings_regularizer= keras.regularizers.l2(1e-5),
#                                     name="user_bias")(user_input)

# movie_bias = keras.layers.Embedding(num_movies,1,
#                                     # embeddings_regularizer= keras.regularizers.l2(1e-5),
#                                     name = "movie_bias")(movie_input)


# # flatten
# user_vec = keras.layers.Flatten()(user_vec)
# movie_vec = keras.layers.Flatten()(movie_vec)
# user_bias = keras.layers.Flatten()(user_bias)
# movie_bias = keras.layers.Flatten()(movie_bias)

# # dot product + biases
# dot = keras.layers.Dot(axes=1)([user_vec, movie_vec])
# # output = keras.layers.Dot(axes=1)([user_vec, movie_vec])


# # output = keras.layers.Add()([dot, user_bias])
# output = keras.layers.Add()([dot, user_bias, movie_bias])


# # build and compile
# model = keras.Model([user_input, movie_input], output)


# model.compile(optimizer=keras.optimizers.Adam(learning_rate = 0.0001),
#               loss = 'mse',
#               metrics=[keras.metrics.RootMeanSquaredError(),'mae'])

# model.summary()

# early_stopping = keras.callbacks.EarlyStopping(
#     monitor='val_loss',
#     patience=15,        # stop if no improvement for 5 epochs
#     restore_best_weights=True  # keep the best model
# )

# # train 
# history = model.fit(
#      [user_arr_s, movie_arr_s],    
#     rating_arr_s,
#     epochs = 150,
#     batch_size = 512,
#     validation_split = 0.2,
#     callbacks=[early_stopping],
#     verbose = 1
# )



# model.save("recommender_model_matrix_factorizartion_v5.keras")


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_3       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_emb            │ (None, 1, 25)     │    148,500 │ input_layer_2[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ movie_emb           │ (None, 1, 25)     │     90,150 │ input_layer_3[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_4 (Flatten) │ (None, 25)        │          0 │ user_emb[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_5 (Flatten) │ (None, 25)        │          0 │ movie_emb[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_bias           │ (None, 1, 1)      │      5,940 │ input_layer_2[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ movie_bias          │ (None, 1, 1)      │      3,606 │ input_layer_3[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dot_1 (Dot)         │ (None, 1)         │          0 │ flatten_4[0][0],  │
│                     │                   │            │ flatten_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_6 (Flatten) │ (None, 1)         │          0 │ user_bias[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_7 (Flatten) │ (None, 1)         │          0 │ movie_bias[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 1)         │          0 │ dot_1[0][0],      │
│                     │                   │            │ flatten_6[0][0],  │
│                     │                   │            │ flatten_7[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 248,196 (969.52 KB)

 Trainable params: 248,196 (969.52 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/150
1217/1217 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.9856 - mae: 0.8253 - root_mean_squared_error: 0.9916 - val_loss: 0.9703 - val_mae: 0.8182 - val_root_mean_squared_error: 0.9839
Epoch 2/150
1217/1217 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.9462 - mae: 0.8058 - root_mean_squared_error: 0.9715 - val_loss: 0.9363 - val_mae: 0.8005 - val_root_mean_squared_error: 0.9665
Epoch 3/150
1217/1217 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.9116 - mae: 0.7875 - root_mean_squared_error: 0.9536 - val_loss: 0.9067 - val_mae: 0.7841 - val_root_mean_squared_error: 0.9510
Epoch 4/150
1217/1217 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.8813 - mae: 0.7704 - root_mean_squared_error: 0.9376 - val_loss: 0.8810 - val_mae: 0.7690 - val_root_mean_squared_error: 0.9374
Epoch 5/150
1217/1217 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.8545 - mae: 0.7549 - root_mean_squared_error: 0.9232 - val_loss: 0.8585 - val_mae: 0.7555 - val_root_mean_squared_error: 0.9253
Epoch 6/150
1217/1217 ━━━━━━━━━━━━━

different terms to address num_factors:

* Embedding dimension 
* Latent dimension
* Number of latent factors (very common in recommender systems)
* Embedding size

Our best Matrix Factorization versions till now

Tried with 25 params

* val_loss(mse)     : 0.6581
* val_mae           : 0.6351
* val_rmse          : 0.8066

Tried with 50 params

* val_loss(mse)     : 0.6767
* val_mae           : 0.6443
* val_rmse          : 0.8160

To get RMSE in the original 1-5 scale:

0.79 × 1.116 ≈ 0.88 stars

val_mae: 0.62 is also in z-score scale:

0.62 × 1.116 ≈ 0.69 stars

### metrics (differences, mean cancels)
rmse_in_stars = rmse_zscore * global_std

### predictions (absolute values, need full reversal)
predicted_stars = (predicted_zscore * global_std) + global_mean

In [29]:
user_emb_matrix = model.get_layer("user_emb").get_weights()[0]
print("user emb std:", np.std(user_emb_matrix))

user_a = user_emb_matrix[1]
user_b = user_emb_matrix[50]
cosine_sim = np.dot(user_a, user_b) / (np.linalg.norm(user_a) * np.linalg.norm(user_b))
print("cosine similarity:", cosine_sim)

user emb std: 0.16866022
cosine similarity: 0.23111904


In [21]:
print("Val RMSE: ", 0.8122*global_std)

Val RMSE:  0.9068145840688058


## how did the model improve from val rmse of 0.98 to 0.81:


1. **Shuffle** (biggest reason)
    
    Data was sorted by UserID. Last 20% was almost entirely users 4750-5940 who never appeared in training. Model had no learned embeddings for them, so it just predicted the global mean for every validation row. Val loss stuck at 1.00.

2. **Z-score normalization instead of just subtracting mean**

    Before, we subtracted mean but didn't divide by std. The target range was roughly [-2.5, 1.4] — asymmetric and wide. After dividing by std, targets are symmetric around 0 with consistent scale. This made the loss landscape smoother and easier to optimize.
    
3. **Removing bias regularization**

    We had l2=1e-5 on both bias embeddings. This was forcing the model to forget real signal like "user 42 always rates 0.8 above average." Once removed, the biases could freely capture per-user and per-movie tendencies, which is a large chunk of what makes good predictions.

4. **Batch size 64 → 512**

    With batch size 64, each gradient update was based on only 64 samples — very noisy estimate of the true gradient. 512 gives a much cleaner signal, so the model takes more reliable steps toward the minimum.

**we gonna keep matrix factorization with 25 latent factors as a baseline models and try to see if we can beat it with a complex model**

## Implementing Neural Collaborative Filtering

It add a Multi Layer Perceptron layer to learn non linear patterns from the embeddings.

In [ ]:

# # SHUFFLE 
# indices = np.random.permutation(len(user_arr))
# user_arr_s   = user_arr[indices]
# movie_arr_s  = movie_arr[indices]
# rating_arr_s = rating_arr_normalized[indices]


# # hyperparameters

# num_users  = len(user_id_map)   
# num_movies = len(movie_id_map)  



# num_factors = 25

# # user inputs
# user_input = keras.Input(shape=(1,))
# movie_input = keras.Input(shape=(1,))

# # embeddings
# user_vec  = keras.layers.Embedding(input_dim = num_users, output_dim = num_factors,
#                                   embeddings_regularizer=keras.regularizers.l2(1e-6),
#                                   embeddings_initializer=keras.initializers.RandomNormal(mean=0, stddev=0.1),
#                                   name= "user_emb")(user_input)

# movie_vec = keras.layers.Embedding(num_movies,num_factors, 
#                                    embeddings_regularizer= keras.regularizers.l2(1e-6),
#                                    embeddings_initializer=keras.initializers.RandomNormal(mean=0, stddev=0.1),
#                                    name="movie_emb")(movie_input)

# # # adding dropout 
# # user_vec = keras.layers.Dropout(0.2)(user_vec)
# # movie_vec = keras.layers.Dropout(0.2)(movie_vec)

# # flatten
# user_vec = keras.layers.Flatten()(user_vec)
# movie_vec = keras.layers.Flatten()(movie_vec)


# # Matrix Factorization
# mf = keras.layers.Dot(axes=1)([user_vec, movie_vec])


# # MLP Path - adding a neural network to learn
# concat = keras.layers.Concatenate()([user_vec,movie_vec])

# mlp = keras.layers.Dense(64, activation='tanh')(concat)
# mlp = keras.layers.Dense(32, activation='tanh')(mlp)
# mlp = keras.layers.Dense(16, activation='tanh')(mlp)


# # combine both paths
# combined = keras.layers.Concatenate()([mf,mlp])


# output = keras.layers.Dense(1)(combined)

# # build and compile
# model = keras.Model([user_input, movie_input], output)


# model.compile(optimizer=keras.optimizers.Adam(learning_rate = 0.0001),
#               loss = 'mse',
#               metrics=[keras.metrics.RootMeanSquaredError(),'mae'])

# model.summary()

# early_stopping = keras.callbacks.EarlyStopping(
#     monitor='val_loss',
#     patience=15,        # stop if no improvement for 5 epochs
#     restore_best_weights=True  # keep the best model
# )

# # train 
# history = model.fit(
#      [user_arr_s, movie_arr_s],    
#     rating_arr_s,
#     epochs = 150,
#     batch_size = 512,
#     validation_split = 0.3,
#     callbacks=[early_stopping],
#     verbose = 1
# )


# model.save("recommender_model_matrix_factorizartion_v6.keras")


### Testing the model

In [ ]:
# test_user_idx = np.array(test_df['user_idx'])
# test_movie_idx = np.array(test_df['movie_idx'])
# test_ratings = np.array(test_df['Rating'])

In [ ]:
# predictions = model([test_user_idx,test_movie_idx] ,training=False)

# predictions = (predictions* global_std) + global_mean

In [ ]:

# from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error


# print("NCF Metrics")
# print("r2_score: ",r2_score(test_ratings,predictions))
# print("mean_squared_error: ",mean_squared_error(test_ratings,predictions))
# print("root_mean_squared_error: ",root_mean_squared_error(test_ratings,predictions))
# print("mean_absolute_error: ",mean_absolute_error(test_ratings,predictions))

# print("---")

# print("Matrix Factorization  Metrics")
# print("r2_score: ",r2_score(test_ratings,predictions))
# print("mean_squared_error: ",mean_squared_error(test_ratings,predictions))
# print("root_mean_squared_error: ",root_mean_squared_error(test_ratings,predictions))
# print("mean_absolute_error: ",mean_absolute_error(test_ratings,predictions))

NCF Metrics
* r2_score:  0.36411911249160767
* mean_squared_error:  0.7902418971061707
* root_mean_squared_error:  0.8889555335044861
* mean_absolute_error:  0.6994234323501587


Matrix Factorization  Metrics (25 embedding size)
* r2_score:  0.34315359592437744
* mean_squared_error:  0.8162968754768372
* root_mean_squared_error:  0.9034914970397949
* mean_absolute_error:  0.7128654718399048

**We can see that there's a small improvement with our NCF model**
* r2_score  - 0.021 more than MF
* mse       - 0.026 less than MF
* rmse      - 0.014 less than MF
* mae       - 0.013 less than MF

Since this is a very small improvement, completely negligible, a user would never feel that difference in recommendation quality.

I am gonna go ahead with the Matrix Factorization model (model v5)

Since this is a very small improvement, completely negligible, a user would never feel that difference in recommendation quality.

NCF's advantage typically grows with larger, noisier datasets. 

MovieLens 1M is relatively small and clean, which limits how much the additional complexity of NCF can help.

I am going ahead with the Matrix Factorization model as it gives similar performance with simpler inference

### Saving the elements

In [25]:
# model.save("recommender_model_matrix_factorizartion_v6.keras")

In [ ]:
# ## Saving the id mapping
# import pickle

# with open("user_id_map.pkl", "wb") as f:
#     pickle.dump(user_id_map, f)

# with open("movie_id_map.pkl", "wb") as f:
#     pickle.dump(movie_id_map, f)

# with open("held_out_users.pkl", "wb") as f:
#     pickle.dump(held_out_users, f)
    
# with open("held_out_movies.pkl", "wb") as f:
#     pickle.dump(held_out_movies, f)

# with open("base_ratings.pkl", "wb") as f:
#     pickle.dump(base_ratings, f)


    
# # global mean
# np.save("global_mean.npy", global_mean)

# np.save("global_std.npy", global_std)
    

# Testing the model

**importing the model**

In [43]:
import pickle
import numpy as np

In [44]:
model = tf.keras.models.load_model('recommender_model_matrix_factorizartion_v5.keras')

In [45]:
# model.layers

In [46]:
model = tf.keras.models.load_model('recommender_model_matrix_factorizartion_v5.keras')

user_emb   = model.get_layer('user_emb')
movie_emb  = model.get_layer('movie_emb')
user_bias  = model.get_layer('user_bias')
movie_bias = model.get_layer('movie_bias')

user_emb_matrix   = user_emb.get_weights()[0]
movie_emb_matrix  = movie_emb.get_weights()[0]
user_bias_matrix  = user_bias.get_weights()[0]
movie_bias_matrix = movie_bias.get_weights()[0]


global_mean = np.load('global_mean.npy')
global_std = np.load('global_std.npy')

with open('user_id_map.pkl', 'rb') as file:
    user_id_map = pickle.load(file)
    file.close()

with open('movie_id_map.pkl', 'rb') as file:
    movie_id_map = pickle.load(file)
    file.close()


model.predict([user_arr, movie_arr]) vs model([user_arr, movie_arr], training = False) vs Manual Multiplication

In [50]:
# model.predict([user_arr, movie_arr]) 


# predictions
pred_norm = model.predict([user_arr, movie_arr]).flatten()


24331/24331 ━━━━━━━━━━━━━━━━━━━━ 19s 755us/step


In [124]:
# model([user_arr, movie_arr],training = False)


# predictions (still normalized)
pred_norm = model([user_arr, movie_arr],training = False) # [ [1], [2], [3], ... ]

# need to convert to array first to flatten. model() returns tensorflow.python.framework.ops.EagerTensor which cant be flattened
pred_norm = np.array(pred_norm).flatten() #[ 1, 2, 3, ...]

In [137]:
# Manual matrix Multiplication

pred_norm = (( movie_emb_matrix @ user_emb_matrix[0] ) + movie_bias_matrix.flatten() + user_bias_matrix[0].flatten()) * global_std + global_mean
# pred_norm = ( user_emb_matrix @ movie_emb_matrix.T ) + movie_bias_matrix.T + user_bias_matrix 

In [138]:
pred_norm

array([4.19342613, 3.48200763, 3.2761169 , ..., 3.63208738, 4.07097012,
       3.79279545], shape=(3606,))

In [148]:
# ## MODEL V 5


# # convert back to real ratings
# pred = (pred_norm * global_std ) + global_mean

# # true ratings (original, NOT normalized)
# true = rating_arr   # make sure you kept this

# from sklearn.metrics import mean_squared_error, mean_absolute_error
# import numpy as np

# rmse = np.sqrt(mean_squared_error(true, pred))
# mae = mean_absolute_error(true, pred)

# print("RMSE:", rmse)
# print("MAE:", mae)

In [176]:
model = tf.keras.models.load_model('recommender_model_matrix_factorizartion_v5.keras')

user_emb_matrix  = model.get_layer("user_emb").get_weights()[0]
movie_emb_matrix = model.get_layer("movie_emb").get_weights()[0]

# 1. overall std — want > 0.3
print("user emb std:", np.std(user_emb_matrix))
print("movie emb std:", np.std(movie_emb_matrix))

# 2. cosine similarity between random pairs — want close to 0, not close to 1
user_a = user_emb_matrix[100]
user_b = user_emb_matrix[200]
user_c = user_emb_matrix[300]

sim_ab = np.dot(user_a, user_b) / (np.linalg.norm(user_a) * np.linalg.norm(user_b))
sim_ac = np.dot(user_a, user_c) / (np.linalg.norm(user_a) * np.linalg.norm(user_c))
sim_bc = np.dot(user_b, user_c) / (np.linalg.norm(user_b) * np.linalg.norm(user_c))

print("cosine sim 100-200:", sim_ab)
print("cosine sim 100-300:", sim_ac)
print("cosine sim 200-300:", sim_bc)

user emb std: 0.16866022
movie emb std: 0.18800043
cosine sim 100-200: 0.10675887
cosine sim 100-300: -0.36778754
cosine sim 200-300: 0.09907193


## Embedding Health Check

| Metric                    | Value  | Interpretation          |
|---------------------------|--------|-------------------------|
| User embedding std        | 0.166  | Small but acceptable    |
| Movie embedding std       | 0.184  | Small but acceptable    |
| Cosine sim (user 100-200) | 0.127  | Slightly similar        |
| Cosine sim (user 100-300) | -0.570 | Very different — good   |
| Cosine sim (user 200-300) | -0.012 | Nearly orthogonal — good|

Low std is expected with L2 regularization — it keeps magnitudes 
small but embeddings still point in different directions.
This is not embedding collapse. Collapse would show all cosine 
similarities near 1.0, meaning all users look identical to the model.

In [ ]:
new_test_df = test_df[test_df['user_idx'].isin(user_arr) | test_df['movie_idx'].isin(movie_arr)]

test_user_arr = np.array(new_test_df['user_idx'])
test_movie_arr = np.array(new_test_df['movie_idx'])

predictions = model.predict([test_user_arr,test_movie_arr] ).flatten() 

In [ ]:
predictions = [min((max(num,1)),5) for num in predictions]

# true ratings (original, NOT normalized)
true = rating_arr   # make sure you kept this

from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

rmse = np.sqrt(mean_squared_error(true, pred))
mae = mean_absolute_error(true, pred)

print("RMSE:", rmse)
print("MAE:", mae)

In [ ]:
rating_with_name = ratings.merge(movies,on=['MovieID'])[['UserID','MovieID','Title','Rating']]

rating_with_name .head()

# Recommend function

In [ ]:
movie_arr

In [ ]:
np.array(1,)

In [51]:
movie_rating_counts = base_ratings.groupby('movie_idx')['Rating'].count()

In [52]:
def collab_recommend(user_id, num_movies=10, min_num_rating= 300, ):
    
    
    """
    This function first filters out the movie with less ratings to only recommend movies with enough ratings
    
    Gets user_idx for the user_id from input (user_idx - Index of that user's embedding)
    """
    popular_movie_idxs  = movie_rating_counts[movie_rating_counts >= min_num_rating].index.values

    # use only popular movies for recommendations
    popular_movie_arr = popular_movie_idxs.astype('int32')
    
    
    ## Getting User Idx to match with user's embedding's index
    try:
        user_idx = user_id_map[user_id]
    
    except KeyError as e:
        print(f"user id: {user_id} not found")
    
    
    # creating an array, as the model requires ([user_arr, movie_arr]) as input, and  len(user_arr) == len(movie_arr) 
    user_arr = np.full(len(popular_movie_arr) , user_idx )  # len(movie_arr) - how long this array will be; user_idx - num in all the slots created 
    
    
    # Predicting normalized ratings
    predictions = (( movie_emb_matrix @ user_emb_matrix[user_idx] ) + movie_bias_matrix.flatten() + user_bias_matrix[user_idx].flatten())
    
    # predictions =   model([user_arr,popular_movie_arr],training = False)
    
    # # need to convert to array first then flatten(), as model() call returns from TF object that we can't user flatten() on    
    # predictions =   np.array(predictions).flatten() # [ 1, 2, 3, ...]

    # convert back to real ratings
    predictions = (predictions * global_std ) + global_mean

    
    # sorting based on rating, enumerate to keep track of the rating's index
    predictions_with_indexes = sorted(
        zip(popular_movie_idxs, predictions),
        key=lambda x: x[1],
        reverse=True
    )
    
    # extracting idx 
    recommend_movie_idx = [idx for idx,rating in predictions_with_indexes[:5]]

    # fetching actual movie_id using idx-to-MovieId map
    recommend_movie_ids  = [idx2movie[idx] for idx in recommend_movie_idx]
    
    

    return recommend_movie_ids
   

In [53]:
for i in range(1,6):
    print('--'*20)
    display(movies.set_index('MovieID').loc[collab_recommend(i)].reset_index())

----------------------------------------


,MovieID,Title,Genres,avg_rating
0,1921,Pi (1998),Sci-Fi|Thriller,3.791191
1,1276,Cool Hand Luke (1967),Comedy|Drama,4.253763
2,1591,Spawn (1997),Action|Adventure|Sci-Fi|Thriller,2.621053
3,3087,Scrooged (1988),Comedy,3.478682
4,2112,Grand Canyon (1991),Crime|Drama,3.488439


----------------------------------------


,MovieID,Title,Genres,avg_rating
0,1217,Ran (1985),Drama|War,4.268908
1,2112,Grand Canyon (1991),Crime|Drama,3.488439
2,1179,"Grifters, The (1990)",Crime|Drama|Film-Noir,3.788701
3,3087,Scrooged (1988),Comedy,3.478682
4,1921,Pi (1998),Sci-Fi|Thriller,3.791191


----------------------------------------


,MovieID,Title,Genres,avg_rating
0,1073,Willy Wonka and the Chocolate Factory (1971),Adventure|Children's|Comedy|Fantasy,3.861386
1,1267,"Manchurian Candidate, The (1962)",Film-Noir|Thriller,4.333333
2,1921,Pi (1998),Sci-Fi|Thriller,3.791191
3,2953,Home Alone 2: Lost in New York (1992),Children's|Comedy,2.392962
4,3107,Backdraft (1991),Action|Drama,3.440736


----------------------------------------


,MovieID,Title,Genres,avg_rating
0,1073,Willy Wonka and the Chocolate Factory (1971),Adventure|Children's|Comedy|Fantasy,3.861386
1,2953,Home Alone 2: Lost in New York (1992),Children's|Comedy,2.392962
2,1921,Pi (1998),Sci-Fi|Thriller,3.791191
3,1217,Ran (1985),Drama|War,4.268908
4,2599,Election (1999),Comedy,3.930355


----------------------------------------


,MovieID,Title,Genres,avg_rating
0,2622,"Midsummer Night's Dream, A (1999)",Comedy|Fantasy,3.349398
1,2599,Election (1999),Comedy,3.930355
2,1960,"Last Emperor, The (1987)",Drama|War,3.976415
3,1073,Willy Wonka and the Chocolate Factory (1971),Adventure|Children's|Comedy|Fantasy,3.861386
4,2501,October Sky (1999),Drama,4.137755


In [22]:
from recommender import collab_recommend

In [23]:
collab_recommend(4)

[np.int64(1109),
 np.int64(254),
 np.int64(2558),
 np.int64(803),
 np.int64(514),
 np.int64(310),
 np.int64(709),
 np.int64(2132),
 np.int64(1067),
 np.int64(690),
 np.int64(1849),
 np.int64(1144),
 np.int64(1105),
 np.int64(1121),
 np.int64(844),
 np.int64(852),
 np.int64(1107),
 np.int64(2204),
 np.int64(1134),
 np.int64(50),
 np.int64(1149),
 np.int64(1840),
 np.int64(1),
 np.int64(1118)]

In [25]:
import pandas as pd

df = pd.read_csv('display_movie_data.csv')

In [27]:
data = collab_recommend(1)

In [28]:
data

[np.int64(514),
 np.int64(355),
 np.int64(444),
 np.int64(844),
 np.int64(1849),
 np.int64(1109),
 np.int64(580),
 np.int64(803),
 np.int64(3342),
 np.int64(310),
 np.int64(3278),
 np.int64(576),
 np.int64(467),
 np.int64(2204),
 np.int64(1108),
 np.int64(972),
 np.int64(254),
 np.int64(1085),
 np.int64(1171),
 np.int64(2711),
 np.int64(3204),
 np.int64(34),
 np.int64(755),
 np.int64(893)]

In [26]:
df[collab_recommend(4)]

KeyError: "None of [Index([1109,  254, 2558,  803,  514,  310,  709, 2132, 1067,  690, 1849, 1144,\n       1105, 1121,  844,  852, 1107, 2204, 1134,   50, 1149, 1840,    1, 1118],\n      dtype='int64')] are in the [columns]"